# 06 · Hypothesentest — Favorita

## 0 · Imports & Setup

In [ ]:
# ════════════════════════════════════════════════════════
# Deep Learning Hyperparameter Tuning – Favorita
# Optimiert für GPU (Kaggle / Colab)
# ════════════════════════════════════════════════════════

# !pip install neuralforecast ray[tune] optuna -q

import os
import numpy as np
import pandas as pd
import polars as pl
import torch
import time
import matplotlib.pyplot as plt

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST, NHITS
from neuralforecast.losses.pytorch import MAE, MSE, HuberLoss
from sklearn.metrics import mean_absolute_error, mean_squared_error

print(f'GPU verfügbar: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

HORIZON  = 28   # aus config
LOOKBACK = 364  # aus config

# ════════════════════════════════════════════════════════
# Daten laden (Kaggle-Pfad anpassen)
# ════════════════════════════════════════════════════════

# Kaggle: Datensatz ist direkt verfügbar unter /kaggle/input/
# Colab:  Parquet-Dateien hochladen oder von Google Drive mounten

PROCESSED = '/kaggle/input/favorita-processed/'  # Pfad anpassen

train_nf = pd.read_parquet(PROCESSED + 'train_nf.parquet')
val_nf   = pd.read_parquet(PROCESSED + 'val_nf.parquet')
static_df = pd.read_parquet(PROCESSED + 'static_df.parquet')

print(f'Train: {train_nf.shape}')
print(f'Val:   {val_nf.shape}')
print(f'Columns: {train_nf.columns.tolist()}')

# ════════════════════════════════════════════════════════
# Hilfsfunktionen
# ════════════════════════════════════════════════════════

def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_nf(preds_df, y_true_df, model_col):
    merged = preds_df.merge(y_true_df, on=['unique_id', 'ds'], how='inner')
    merged = merged.dropna(subset=[model_col, 'y'])
    y_true = merged['y'].values
    y_pred = merged[model_col].values
    return {
        'MAE':  mean_absolute_error(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'MAPE': mape(y_true, y_pred),
    }

# ════════════════════════════════════════════════════════
# EXPERIMENT 1: PatchTST – Basis vs. getunt
# ════════════════════════════════════════════════════════

patchtst_configs = {
    'PatchTST_base': PatchTST(
        h=HORIZON,
        input_size=LOOKBACK,
        patch_len=16,
        stride=8,
        encoder_layers=2,
        n_heads=8,
        hidden_size=64,
        linear_hidden_size=128,
        dropout=0.2,
        fc_dropout=0.2,
        scaler_type='standard',
        max_steps=200,
        batch_size=64,
        learning_rate=1e-4,
        loss=MAE(),
    ),
    'PatchTST_tuned': PatchTST(
        h=HORIZON,
        input_size=LOOKBACK,
        patch_len=24,        # größere Patches für wöchentliche Muster
        stride=12,
        encoder_layers=3,    # tieferes Netz
        n_heads=16,          # mehr Attention Heads
        hidden_size=128,     # größere Hidden Dimension
        linear_hidden_size=256,
        dropout=0.1,         # weniger Dropout bei mehr Daten
        fc_dropout=0.1,
        scaler_type='standard',
        max_steps=500,       # mehr Training
        batch_size=32,       # kleinere Batches für bessere Generalisierung
        learning_rate=5e-5,  # langsameres Lernen
        loss=MAE(),
        early_stop_patience_steps=20,
    ),
    'PatchTST_large': PatchTST(
        h=HORIZON,
        input_size=LOOKBACK,
        patch_len=7,         # patch = genau eine Woche
        stride=1,
        encoder_layers=4,
        n_heads=8,
        hidden_size=256,
        linear_hidden_size=512,
        dropout=0.15,
        fc_dropout=0.15,
        scaler_type='standard',
        max_steps=500,
        batch_size=32,
        learning_rate=1e-4,
        loss=HuberLoss(),    # robuster gegenüber Ausreißern
        early_stop_patience_steps=20,
    ),
}

results_patchtst = {}
times_patchtst = {}

for name, model in patchtst_configs.items():
    print(f'\n{"═"*50}')
    print(f'Training: {name}')
    print(f'{"═"*50}')
    t0 = time.time()
    nf = NeuralForecast(models=[model], freq='D')
    nf.fit(df=train_nf)
    preds = nf.predict().reset_index()
    elapsed = time.time() - t0
    times_patchtst[name] = elapsed
    results_patchtst[name] = evaluate_nf(preds, val_nf, name.split('_')[0] + '-median')
    print(f'Zeit: {elapsed:.0f}s | Ergebnis: {results_patchtst[name]}')

print('\n── PatchTST Ergebnisvergleich ──')
print(pd.DataFrame(results_patchtst).T.round(2))

# ════════════════════════════════════════════════════════
# EXPERIMENT 2: NHITS – Basis vs. getunt
# ════════════════════════════════════════════════════════

hist_exog = [c for c in train_nf.columns
             if c not in ['unique_id', 'ds', 'y'] and 'static' not in c]
stat_exog = static_df.columns.drop('unique_id').tolist() if static_df is not None else []

nhits_configs = {
    'NHITS_base': NHITS(
        h=HORIZON,
        input_size=LOOKBACK,
        hist_exog_list=hist_exog,
        stat_exog_list=stat_exog,
        stack_types=['identity', 'identity', 'identity'],
        n_blocks=[1, 1, 1],
        mlp_units=[[256, 256]] * 3,
        n_harmonics=0,
        n_polynomials=0,
        dropout_prob_theta=0.2,
        scaler_type='standard',
        max_steps=200,
        batch_size=64,
        learning_rate=1e-3,
        loss=MAE(),
    ),
    'NHITS_tuned': NHITS(
        h=HORIZON,
        input_size=LOOKBACK,
        hist_exog_list=hist_exog,
        stat_exog_list=stat_exog,
        stack_types=['identity', 'identity', 'identity'],
        n_blocks=[2, 2, 2],          # mehr Blöcke je Stack
        mlp_units=[[512, 512]] * 3,  # breitere MLP-Schichten
        n_harmonics=0,
        n_polynomials=0,
        dropout_prob_theta=0.1,
        scaler_type='standard',
        max_steps=500,
        batch_size=32,
        learning_rate=5e-4,
        loss=MAE(),
        early_stop_patience_steps=20,
    ),
    'NHITS_huber': NHITS(
        h=HORIZON,
        input_size=LOOKBACK,
        hist_exog_list=hist_exog,
        stat_exog_list=stat_exog,
        stack_types=['identity', 'identity', 'identity'],
        n_blocks=[2, 2, 2],
        mlp_units=[[512, 512]] * 3,
        dropout_prob_theta=0.1,
        scaler_type='standard',
        max_steps=500,
        batch_size=32,
        learning_rate=5e-4,
        loss=HuberLoss(),   # robust gegenüber Feiertagspeaks
        early_stop_patience_steps=20,
    ),
}

results_nhits = {}
times_nhits = {}

for name, model in nhits_configs.items():
    print(f'\n{"═"*50}')
    print(f'Training: {name}')
    print(f'{"═"*50}')
    t0 = time.time()
    nf = NeuralForecast(models=[model], freq='D')
    nf.fit(df=train_nf, static_df=static_df)
    preds = nf.predict(static_df=static_df).reset_index()
    elapsed = time.time() - t0
    times_nhits[name] = elapsed
    results_nhits[name] = evaluate_nf(preds, val_nf, name.split('_')[0])
    print(f'Zeit: {elapsed:.0f}s | Ergebnis: {results_nhits[name]}')

print('\n── NHITS Ergebnisvergleich ──')
print(pd.DataFrame(results_nhits).T.round(2))

# ════════════════════════════════════════════════════════
# EXPERIMENT 3: Bestes Modell auf Test-Set
# ════════════════════════════════════════════════════════

# Bestes PatchTST und bestes NHITS identifizieren
best_pt  = min(results_patchtst, key=lambda k: results_patchtst[k]['MAE'])
best_nhi = min(results_nhits,    key=lambda k: results_nhits[k]['MAE'])
print(f'\nBestes PatchTST: {best_pt}  → MAE {results_patchtst[best_pt]["MAE"]:.1f}')
print(f'Bestes NHITS:    {best_nhi} → MAE {results_nhits[best_nhi]["MAE"]:.1f}')

# ════════════════════════════════════════════════════════
# Ergebnisplot: Alle Konfigurationen im Vergleich
# ════════════════════════════════════════════════════════

all_results = {**results_patchtst, **results_nhits}
results_plot = pd.DataFrame(all_results).T.reset_index().rename(columns={'index': 'Modell'})

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = (
    ['#4C72B0'] * len(results_patchtst) +
    ['#55A868'] * len(results_nhits)
)
for i, metric in enumerate(['MAE', 'RMSE', 'MAPE']):
    axes[i].bar(results_plot['Modell'], results_plot[metric],
                color=colors, edgecolor='none')
    axes[i].set_title(f'{metric} – Val-Set', fontsize=12)
    axes[i].tick_params(axis='x', rotation=40)
    axes[i].set_ylabel(metric)
plt.suptitle('Hyperparameter-Tuning: PatchTST vs. NHITS – Val-Set', fontsize=13)
plt.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════
# Export: Beste Predictions für NB05
# ════════════════════════════════════════════════════════

# train_val_nf = pd.concat([train_nf, val_nf])  # für Test-Set Retrain

# nf_best = NeuralForecast(models=[patchtst_configs[best_pt]], freq='D')
# nf_best.fit(df=train_val_nf)
# test_preds_pt = nf_best.predict().reset_index()
# test_preds_pt.to_parquet('test_patchtst_tuned.parquet')

# nf_best_nhi = NeuralForecast(models=[nhits_configs[best_nhi]], freq='D')
# nf_best_nhi.fit(df=train_val_nf, static_df=static_df)
# test_preds_nhi = nf_best_nhi.predict(static_df=static_df).reset_index()
# test_preds_nhi.to_parquet('test_nhits_tuned.parquet')

print('\nExport-Zellen auskommentiert – aktivieren sobald beste Configs feststehen.')
print('Parquet-Dateien dann in NB05 einlesen.')